In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import sys
from datetime import datetime

In [3]:
places_csv = pd.read_csv("//depot.engr.oregonstate.edu/mime_u1/dalziel/Safe Graph Data/Weekly Patterns/Digital_Twins_Analysis/Places/2019.csv")

monthly = "//depot.engr.oregonstate.edu/mime_u1/dalziel/Safe Graph Data/Weekly Patterns/Digital_Twins_Analysis/2019/2019_monthly/Newport"
weekly = "//depot.engr.oregonstate.edu/mime_u1/dalziel/Safe Graph Data/Weekly Patterns/Digital_Twins_Analysis/2019/2019_weekly/Newport"

In [4]:
monthly_dfs = [pd.read_csv(path) for path in [os.path.join(monthly, item) for item in os.listdir(monthly)]]
# currently truncated to the first 23 weeks because only 4 intermediate monthly files have been created
weekly_dfs = [pd.read_csv(path) for path in [os.path.join(weekly, item) for item in os.listdir(weekly)[:23]]]

In [5]:
monthly_melted = []
for df in monthly_dfs:
    df['month'] = pd.to_datetime(df['date_range_start']).dt.month
    df = df.melt(
        id_vars = ['safegraph_place_id', 'median_dwell', 'month'],
        value_vars = [f"visits_h{i}" for i in range(24)],
        var_name = "hour_of_day",
        value_name = "dwell"
    )
    df['hour_of_day'] = df['hour_of_day'].str.extract(r'(\d+)').astype(int)
    monthly_melted.append(df)

In [6]:
monthly_melted[1]

,safegraph_place_id,median_dwell,month,hour_of_day,dwell
0,sg:d6ee22e287fc4a29b5bcb55053973bba,108.0,1,0,5
1,sg:a711c7a6638b44b4937cdd2177ee212d,8.0,1,0,0
2,sg:90be272ba24d4e7ea06aed27bae751f7,33.0,1,0,5
3,sg:e488c8e89a494897b452ab7432c56f30,53.0,1,0,0
4,sg:4faea969e4b34305a716ec3341a3b422,16.0,1,0,2
...,...,...,...,...,...
9331,sg:f854e584257746458832c2beb2884bba,11.0,1,23,0
9332,sg:90cc0e7ed6ab4201b6c1917a4beefb91,67.0,1,23,76
9333,sg:5a379e6242bc4846a3afbc25a5b679c1,18.0,1,23,11
9334,sg:931db17bc2a845a3a73610bc022e9b09,29.0,1,23,5


In [7]:
massive_df = pd.concat(weekly_dfs, ignore_index = True)

In [8]:
massive_df.head()
# 11k columns...

,safegraph_place_id,raw_visit_counts,raw_visitor_counts,poi_cbg,distance_from_home,median_dwell,visits_h0,visits_h1,visits_h2,visits_h3,...,d_cbg:132950208003,d_cbg:060855046021,d_cbg:410279502002,d_cbg:530530714061,d_cbg:410279504003,d_cbg:410579605002,d_cbg:040210017092,d_cbg:410290011003,d_cbg:530150013004,d_cbg:060014503002
0,sg:83dd7c5b0881400f8b2abcb77c5347bd,5,5,410419511001,5577,50.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sg:0ae6746f4222414ba2bc87422537c351,128,105,410419510002,109219,23.5,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sg:79fcd454a98c41deb79c9fe68d917fa4,84,52,410419509002,13457,11.5,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sg:ac4240b60fff40af824309a1557cb020,96,66,410419509004,11809,18.5,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sg:9973283350f84b84a0bb5b3862f916a3,61,56,410419510002,141774,6.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
weekly_melted = massive_df.melt(
    id_vars = ['safegraph_place_id', 'median_dwell', 'distance_from_home'],
    value_vars=[f"visits_h{i}" for i in range(168)], 
    var_name='hour_of_week', 
    value_name='arrivals'
)
weekly_melted['hour_of_week'] = weekly_melted['hour_of_week'].str.extract(r'(\d+)').astype(int)
weekly_melted['hour_of_day'] = weekly_melted['hour_of_week']%24

weekly_melted.head(5)

,safegraph_place_id,median_dwell,distance_from_home,hour_of_week,arrivals,hour_of_day
0,sg:83dd7c5b0881400f8b2abcb77c5347bd,50.0,5577,0,0,0
1,sg:0ae6746f4222414ba2bc87422537c351,23.5,109219,0,0,0
2,sg:79fcd454a98c41deb79c9fe68d917fa4,11.5,13457,0,0,0
3,sg:ac4240b60fff40af824309a1557cb020,18.5,11809,0,0,0
4,sg:9973283350f84b84a0bb5b3862f916a3,6.0,141774,0,0,0


In [15]:
weekly_agg = weekly_melted.groupby(['safegraph_place_id','hour_of_day']).agg({'arrivals':'sum', 'distance_from_home': 'min', 'median_dwell': 'min'})
weekly_agg

arrivals  distance_from_home  \
safegraph_place_id                  hour_of_day                                 
sg:001e3da6479b40d389529752513d2657 0                   0                  -1   
                                    1                   0                  -1   
                                    2                   0                  -1   
                                    3                   0                  -1   
                                    4                   0                  -1   
...                                                   ...                 ...   
sg:ffadd5fae8c64045b8199d5d4c1838d0 19                  0                  -1   
                                    20                  0                  -1   
                                    21                  0                  -1   
                                    22                  0                  -1   
                                    23                  1                  -1   

                                                 median_dwell  
safegraph_place_id                  hour_of_day                
sg:001e3da6479b40d389529752513d2657 0                     5.0  
                                    1                     5.0  
                                    2                     5.0  
                                    3                     5.0  
                                    4                     5.0  
...                                                       ...  
sg:ffadd5fae8c64045b8199d5d4c1838d0 19                    6.0  
                                    20                    6.0  
                                    21                    6.0  
                                    22                    6.0  
                                    23                    6.0  

[9936 rows x 3 columns]